# LearnRAG: a simple local RAG pipeline

This notebook retrieves relevant document passages and gives them to a local language model to generate an answer. It uses public models with no paid API key. Initial model downloads require Internet access and several GB of space. A Colab GPU is recommended.

Real-model outputs have not been verified in the preparation environment because Hugging Face access returned HTTP 403. Component tests use test doubles and do not measure answer quality.


## 1. Get the code and install dependencies

In Colab, choose **Runtime → Change runtime type → T4 GPU** if available. The next cell clones LearnRAG only if the current folder does not already contain its script. Existing files are reused; it does not pull or overwrite changes.


In [ ]:
from pathlib import Path
import os, subprocess

if not Path("simple_rag.py").is_file():
    project = Path.cwd() / "LearnRAG"
    if not project.exists():
        subprocess.run(["git", "clone", "https://github.com/nafismam/LearnRAG.git", str(project)], check=True)
    if not (project / "simple_rag.py").is_file():
        raise RuntimeError("LearnRAG exists but does not contain simple_rag.py; select the correct project directory.")
    os.chdir(project)
print("Project directory:", Path.cwd())


In [ ]:
%pip install -r requirements.txt


## 2. Inspect the data and chunks
Chunks retain a relative file path and a unique ID. Generic character windows are used here, rather than statutory section extraction.


In [ ]:
from simple_rag import RAGConfig, SimpleRAG, load_documents, chunk_documents
config = RAGConfig(temperature=0.0)
docs = load_documents(config.data_dir)
chunks = chunk_documents(docs, config.chunk_size, config.chunk_overlap)
print("Chunks:", len(chunks))
print(chunks[0])


## 3. Build the index and load the generator
MiniLM embeds chunks into normalized vectors stored in NumPy. Qwen generates answers from retrieved context. The first run downloads several GB; a Colab GPU is recommended. CPU generation can be slow.


In [ ]:
rag = SimpleRAG(config)


## 4. Retrieve, augment and generate
Inspect the actual source text alongside each answer. Numbered citations are model output, not automatically verified evidence.


In [ ]:
result = rag.ask("Which embedding model does this pipeline use?")
print(result["sources"])


In [ ]:
rag.ask("What is Retrieval-Augmented Generation and why is it useful?")


## 5. Check an unsupported question
The prompt asks the model to abstain when context lacks an answer. This is an observation to inspect, not a guaranteed passing test. Retrieval always returns top-k chunks and currently has no confidence gate.


In [ ]:
rag.ask("Who won the 1998 FIFA World Cup?")
